# Human-in-the-Loop: Pre-Action Gates, Risk Tiering, and Audit Logging

Di README for dis lesson introduce Human-in-the-Loop wit small snippet wey dey ask di user `APPROVE` or `REJECT` after di agent don already produce response. Dat pattern na beta starting point, but production HITL implementations dey usually need three more things:

1. One **pre-action gate** wey dey run **before** di agent perform risky step, so cost, wahala wey no fit reverse, and delay go still dey inside control.
2. **Risk tiering**, so say low-risk actions go run automatic, medium-risk actions go dey batch-approved, and only high-risk actions go hold for human approval.
3. One **audit log plus revision loop**, so every gate decision go dey recorded as JSONL, and when them reject, e go re-prompt agent wit structured reason instead of just print `Revising...`.

Dis notebook go build each of these top of di same basics as `06-system-message-framework.ipynb`. E go run full end-to-end for `DEMO_MODE = True` (no need to input anything) or wit real `input()` prompts when `DEMO_MODE = False`. Note: for DEMO_MODE, di retry for di third goal na script so di loop mechanics fit show clear clear end-to-end. Real revision-driven re-classification need `DEMO_MODE = False` plus operator.

**No cover here (dem dey for oda lessons):** authentication and access control (Lesson 06 README threat 2), tool-call middleware (Lesson 14 MAF deep dive), multi-agent debate patterns.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Pattern 1: Pre-action gate

The README's HITL snippet dey call di agent first, den e ask di user make e approve di output. Dat na **post-action** flow. Di agent don already execute, so di LLM call cost don already pay, and any side effect (send email, write database row, post comment) don already happen.

One **pre-action** flow go put di gate before di agent run di risky step. Di agent go propose di action, di gate go decide if e go execute, and na only if e approve na di side effect go happen.

| Aspect | Post-action approval (README snippet) | Pre-action gate (this notebook) |
|---|---|---|
| When does approval run? | After di agent don produce output | Before any side-effect execute |
| LLM cost on rejection | Don already pay | Na only for di proposal e pay, no be for di action |
| Irreversible side effects | E fit happen (di action don already happen) | Dem go prevent am |
| Audit clarity | Approval na print statement | Approval na JSON record with timestamp, action, reason |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

No evri action need human approval. A read-only lookup against a public API get different risk than to send customer email. To treat both the same na to waste operator attention and e go slow down the agent.

Simple 3-tier model: 

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search knowledge base, check flight options, fetch public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache result, add counter, arrange reminder | Auto-execute, but dem go review am once daily |
| `high` (external-facing or irreversible) | Send email, charge card, post to public channel | Wait for human approval |

Dis na one kind tiering. Production systems dey use more detailed tiers (like AWS IAM permission levels, role-based access tiers). The 3-tier we get below na the smallest wey make sense for agent wey dey do read-only and side-effect actions.

The classifier below use keyword heuristics so the demo go remain deterministic and cheap. For production, you fit replace am with learned classifier or policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Audit log and revision loop

One `print("Response approved.")` no be audit log. For trust sake, every gate decision suppose dey recorded as structured event wey you fit later check, play back, or join with incident review.

Two parts:

1. **Append-only JSONL.** One line per decision, get timestamp, action, tier, decision, reason. E easy to grep, e easy to send go real log store later.
2. **Revision loop on rejection.** When gate talk `deny`, the agent go re-ask itself with the rejection reason inside, so the next proposal no go carry the same wahala.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Mo resos wey fit help you

Plenti oda public projek dem dey implement diffren kain HITL patan dem. Compare di kain way dem dey take do am make you sabi wetin go match your stack:

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): na tool wrappers wey fit pause to make human fit put input.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ don rearrange am): e dey use agent role make e represent human inside multi-agent conversation.
- **Microsoft Agent Framework (MAF)** function-invocation middleware ([docs](https://learn.microsoft.com/agent-framework/)): na middleware wey dey run for every tool/function call, e good to control logic and approval flow.

Every projek dey handle the three sub-patan dem different: LangChain na tool wrapper, AutoGen na agent role, Microsoft Agent Framework na function-invocation middleware. Try read one or two implementation finish finish before you choose design wey go work for your own agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
